In [1]:
import pandas as pd
import jieba
from gensim.models.word2vec import Word2Vec

D:\Users\33097\miniconda3\envs\nlp_dyj\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
D:\Users\33097\miniconda3\envs\nlp_dyj\lib\site-packages\gensim\utils.py:35: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 1.22.3)
  import scipy.sparse


In [2]:
data = pd.read_csv('train.csv')
print(data.head())

   label                                            comment
0      0                                 一如既往地好吃，希望可以开到其他城市
1      0                                  味道很不错，分量足，客人很多，满意
2      0  下雨天来的，没有想象中那么火爆。环境非常干净，古色古香的，我自己也是个做服务行业的，我都觉得...
3      0                                    真心不好吃 基本上没得好多味道
4      0              少送一个牛肉汉堡 而且也不好吃 特别是鸡肉卷 **都不想评论了 谁买谁知道


In [3]:
corpus = data['comment'].values.astype(str)

corpus = [jieba.lcut(corpus[index]
                          .replace("，", "")
                          .replace("!", "")
                          .replace("！", "")
                          .replace("。", "")
                          .replace("~", "")
                          .replace("；", "")
                          .replace("？", "")
                          .replace("?", "")
                          .replace("【", "")
                          .replace("】", "")
                          .replace("#", "")
                        ) for index in range(len(corpus))]

print(corpus[:2])  # 查看前两条分词结果

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\33097\AppData\Local\Temp\jieba.cache
Loading model cost 0.329 seconds.
Prefix dict has been built successfully.


[['一如既往', '地', '好吃', '希望', '可以', '开', '到', '其他', '城市'], ['味道', '很', '不错', '分量', '足', '客人', '很多', '满意']]


In [4]:
model = Word2Vec(
    corpus,
    sg=1,
    vector_size=300,
    window=5,
    min_count=3,
    workers=4
)

print('模型参数：', model)

模型参数： Word2Vec<vocab=4036, vector_size=300, alpha=0.025>


In [5]:
vector = model.wv["环境"]
print("环境 的词向量：\n", vector)
print("向量形状：", vector.shape)

环境 的词向量：
 [ 5.62167652e-02  1.70201138e-01  3.46231163e-02  1.25368834e-01
  1.25307006e-05 -2.82067925e-01 -4.05528434e-02  4.74045962e-01
 -2.99294025e-01 -3.03791035e-02  7.65963569e-02 -2.81929433e-01
 -8.74981284e-02  9.40384269e-02 -1.04726136e-01 -1.45258114e-01
  5.17233253e-01  1.57047942e-01  2.95505762e-01 -3.44311744e-01
 -7.34453425e-02 -2.63093889e-01 -5.94575964e-02  3.66051719e-02
 -2.36305326e-01  7.23225027e-02 -1.43702114e-02 -4.40497845e-02
 -6.17982969e-02 -2.54798591e-01  3.40405196e-01 -1.41161270e-02
  1.30573377e-01  1.88601270e-01 -1.78480402e-01  7.19435420e-03
 -1.42830387e-01 -1.16504230e-01  5.99300303e-02 -3.82864363e-02
  1.83317602e-01 -1.31234840e-01  1.81633189e-01 -1.20799087e-01
  3.19536030e-01  1.86093539e-01 -9.34385285e-02 -1.45855621e-01
  1.00541592e-01  2.07445454e-02  5.18909916e-02 -4.74234037e-02
 -2.28647783e-01  2.69450605e-01  1.88239571e-02  1.33642212e-01
 -1.41521975e-01 -2.04200670e-01  1.49163231e-01 -1.73294365e-01
 -2.91276742e-0

In [6]:
similar_words = model.wv.most_similar("好吃", topn=3)
print("与“好吃”最接近的3个词：")
print(similar_words)

与“好吃”最接近的3个词：
[('奶油', 0.8281220197677612), ('入味', 0.8276149034500122), ('棒', 0.8243154287338257)]


In [7]:
sim1 = model.wv.similarity("好吃", "美味")
sim2 = model.wv.similarity("好吃", "蟑螂")

print("好吃 vs 美味 相似度：", sim1)
print("好吃 vs 蟑螂 相似度：", sim2)

好吃 vs 美味 相似度： 0.79785216
好吃 vs 蟑螂 相似度： 0.28627518


In [8]:
result = model.wv.most_similar(
    positive=["餐厅", "聚会"],
    negative=["安静"],
    topn=1
)

print("餐厅 + 聚会 - 安静 ≈ ", result)

餐厅 + 聚会 - 安静 ≈  [('亲人', 0.9418426752090454)]
